[Guide Index](README.md)


# SPARQL-RL Rules as RDF — User Guide

A parsed SRL rule set — `srl.RuleSet`, the same dataclass tree `srl.parse_ruleset()` returns (see the [SPARQL-RL rules guide](03b-sparql-rules.ipynb) for the rules language itself) — can also be represented as an RDF graph, under the `srl:` namespace, the same "AST-as-RDF" pattern used elsewhere in StarLayer (e.g. [SPARQL queries as RDF](05b-sparql-query-as-rdf.ipynb), [Manchester Syntax](05a-manchester-syntax.ipynb)). This lets a rule set be stored, inspected, or edited as ordinary RDF, and validated structurally with SHACL.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb) if not. Run cells from top to bottom; later sections reuse variables from earlier ones.


## 1. A parsed rule set

The same base graph and rule set as the [SPARQL-RL rules guide](03b-sparql-rules.ipynb) - see that guide for what these rules mean; this one is only about representing the already-parsed result as RDF.


In [1]:
from starlayer.graph import StarLayerGraph, Namespace
from starlayer.sparql import srl

EX = Namespace("http://example.org/")

base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")
base.bind("ex", EX)

rules_text = """
PREFIX ex: <http://example.org/>

RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")


4 rules parsed


## 2. Encode the rule set as RDF

StarLayer allows for text-based SRL rules to be translated into a RDF-tree graph under the `srl:` namespace. This allows a rule set to be edited and stored as an RDF graph. `tree_to_ruleset()` decodes the graph into a ruleset.


In [2]:
srl_graph, root = srl.ruleset_to_tree(ruleset)
print("ruleset as graph:")
print(srl_graph.serialize())


ruleset as graph:
@prefix ns1: <https://github.com/hidden-graph/starlayer/ns/sparql/srl#> .
@prefix ns2: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix ns3: <http://example.org/> .
@prefix ns4: <https://github.com/hidden-graph/starlayer/ns/sparql/algebra#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:N93895f2d9f964d0f9c8984c29e1b0b0e a ns1:RuleSet ;
    ns1:data () ;
    ns1:imports () ;
    ns1:rules ( [
        a ns1:Rule ;
        ns1:body ( [
            a ns1:TriplePattern ;
            ns1:object "v"^^ns2:Variable ;
            ns1:predicate ns3:hasVulnerability ;
            ns1:subject "x"^^ns2:Variable
        ] ) ;
        ns1:head ( [
            a ns1:TriplePattern ;
            ns1:object "v"^^ns2:Variable ;
            ns1:predicate ns3:exposedTo ;
            ns1:subject "x"^^ns2:Variable
        ] ) ;
        ns1:ruleData false
    ] [
        a ns1:Rule ;
        ns1:body ( [

## 3. Validate the encoding with SHACL

`srl.srl_validate(data_graph)` checks that an `srl:` graph is structurally well-formed using a `shacl` shape.


In [3]:
conforms, _report_graph, report_text = srl.srl_validate(srl_graph)
print("conforms:", conforms)


conforms: True
